# Notebook 08 - US Case Table
**Purpose** Merges our scraped and labeled FTC cases (from FTC directly) with Rafal's entries into our working US table

**Inputs**
- `data/ftc_labeled.csv` (339 cases)
- `data/ftc_rafal_cases.csv` (285 entries)

**Outputs**
- `data/us_cases.csv` (339 x 39)

**Decisions made**
- Rafal labels are primary and treated as base for the aggregate table, Rafal: 274 cases, ours: 65 cases, source recorded in attribute 'label source'
- Rafal had 9 URLs that accounted for 19 rows, these were collapsed to 9 rows, bringing his 285 entries to 275,
- 1 entry lacked 'statutory_topics' 2019 Facebook consent order, thus 274 entries
- - this case is accounted for in our scrape
- 5 superceded/redundant columns pruned, full lineage still available upstream

**Gaps/Shortcomings**
- 3 suspected URL misassignments in Rafal (Genica, 214 Technologies, AYLO)
- 13 label disagreements flagged via audit
- 17 cases to be ajudicated by hand, labeled for later

**Last run:** 2026-07-14 — 339 rows, agreements .989/.978/.985
- COPPA: 271/274 match = .989 agreement
- GLBA: 268/274 match = .978 agreement
- FCRA: 270/274 match = .985 agreement
- **worth noting** this is agreement between ftc scraped and Rafal, not necessarily ground truth

In [9]:
import pandas as pd

In [10]:
us = pd.read_csv("/Users/nic/Documents/MM2/data/ftc_labeled.csv")
raf = pd.read_csv("/Users/nic/Documents/MM2/data/ftc_rafal_cases.csv")

us["url"] = us["url"].str.rstrip("/")
raf["ftc_url"] = raf["ftc_url"].str.rstrip("/")
raf = raf.groupby("ftc_url", as_index=False).agg(lambda s: "; ".join(sorted(set(str(v) for v in s.dropna()))) if s.dtype == object else s.iloc[0])
raf = raf.add_prefix("rafal_")

merged = us.merge(raf, left_on="url", right_on="rafal_ftc_url", how="left")
print(merged.shape)
print("cases with Rafal labels:", merged["rafal_statutory_topics"].notna().sum())


(339, 42)
cases with Rafal labels: 274


In [11]:
merged["label_source"] = merged["rafal_statutory_topics"].notna().map({True: "rafal", False: "ours"})
merged["violation_labels_final"] = merged["rafal_statutory_topics"].fillna(merged["violation_labels"])
print(merged["label_source"].value_counts())

label_source
rafal    274
ours      65
Name: count, dtype: int64


In [12]:
for statute, tagkey in [("COPPA", "COPPA"), ("GLBA", "Gramm-Leach-Bliley"), ("FCRA", "Credit Reporting")]:
    both = merged[merged["label_source"] == "rafal"]
    his = both["rafal_statutory_topics"].str.contains(statute, na=False)
    ours = both["tags"].str.contains(tagkey, na=False)
    print(f"{statute}: agreement {(his == ours).mean():.3f} | disagreements: {both.loc[his != ours, 'case_name'].tolist()}")

##Pruning superceded attributes
drop_cols = ["statutes", "statutes_recovered", "penalty_usd", "rafal_ftc_url", "case_status",]
merged = merged.drop(columns=drop_cols)
print(f"dropped {len(drop_cols)} superceded/legacy columns, {merged.shape[1]} remain")

merged.to_csv("/Users/nic/Documents/MM2/data/us_cases.csv", index=False)
print(merged.shape)

COPPA: agreement 0.989 | disagreements: ['Microsoft Corporation, U.S. v.', 'Miniclip, In the Matter of', 'Retina-X Studios, LLC, In the Matter of']
GLBA: agreement 0.978 | disagreements: ['Equifax, Inc.', "Franklin's Budget Car Sales, Inc., also d/b/a Franklin Toyota/Scion, In the Matter of", 'Action Research Group, Inc., et al.', 'Goal Financial, LLC, In the Matter of', 'CEO Group, Inc. d/b/a Check Em Out, and Scott Joseph', 'Integrity Security & Investigation Services, Inc.']
FCRA: agreement 0.985 | disagreements: ['ITMedia Solutions LLC', 'BoostMyScore LLC', 'Sitesearch Corporation, Doing Business As LeapLab', 'Consumerinfo.com., Inc., d/b/a Experian Consumer Direct, Qspace, Inc., and Iplace Inc.']
dropped 5 superceded/legacy columns, 39 remain
(339, 39)


In [13]:
##Previews the final table
import pandas as pd
us = pd.read_csv("/Users/nic/Documents/MM2/data/us_cases.csv")
us.sample(3, random_state=1).T

,102,125,11
case_name,"VenPath, Inc., In the Matter of","Turn Inc., In the Matter of",Apitor
url,https://www.ftc.gov/legal-library/browse/cases...,https://www.ftc.gov/legal-library/browse/cases...,https://www.ftc.gov/legal-library/browse/cases...
date,2018-11-19,2017-04-21,2025-10-01
case_type,Administrative,Administrative,Federal
matter_number,1823144,1523099,NaN
summary,NaN,NaN,The FTC reached a settlement with Apitor Techn...
tags,Consumer Protection; international cooperation...,Consumer Protection; Office of Technology Rese...,Consumer Protection; Regional Offices; Bureau ...
long_title,"In the Matter of VenPath, Inc., a corporation.","In the Matter of Turn Inc., a corporation.","United States of America, Plaintiff, v. Apitor..."
press_release_urls,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/news-events/news/press-rel...,https://www.ftc.gov/news-events/news/press-rel...
pdf_urls,https://www.ftc.gov/system/files/documents/cas...,https://www.ftc.gov/system/files/documents/cas...,https://www.ftc.gov/system/files/ftc_gov/pdf/A...
